In [2]:
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

# --- 1. Create Parallel Environments ---
# This is the most important change.
# We create 16 "Hopper" environments to run in parallel on the CPU.
# We remove render_mode="human" to stop the slow rendering.
print("--- 1. Creating Vectorized Environment ---")
env = make_vec_env("Hopper-v4", n_envs=16)
# env = make_vec_env("Hopper-v4", render_mode="human")

# --- 2. Create PPO Agent on the GPU ---
# We add device="cuda" to tell the model to use the GPU.
# We increase the batch_size to give the GPU more work to do at once,
# which is more efficient.
print("--- 2. Creating PPO Agent on GPU ---")
model = PPO(
    "MlpPolicy",
    env,
    verbose=1,
    device="cuda",      # <-- Tell it to use the GPU
    batch_size=512,     # <-- Give the GPU a bigger chunk of work
    n_steps=4096        # <-- Collect more data before each update
)

# --- 3. Training Agent ---
# This will now be much faster and you'll see higher GPU-Util
print("--- 3. Training Agent ---")
model.learn(total_timesteps=10000000) # Increased steps, as it's faster now

print("--- 4. Saving Trained Model ---")
model.save("ppo_hopper_gpu_ipynb")

# Clean up the environments
env.close()

--- 1. Creating Vectorized Environment ---
--- 2. Creating PPO Agent on GPU ---
Using cuda device
--- 3. Training Agent ---


/home/jamesspectre007/anaconda3/envs/forcelearning/lib/python3.13/site-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment Hopper-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 21.3     |
|    ep_rew_mean     | 17.9     |
| time/              |          |
|    fps             | 3650     |
|    iterations      | 1        |
|    time_elapsed    | 17       |
|    total_timesteps | 65536    |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 29.5        |
|    ep_rew_mean          | 31.9        |
| time/                   |             |
|    fps                  | 3685        |
|    iterations           | 2           |
|    time_elapsed         | 35          |
|    total_timesteps      | 131072      |
| train/                  |             |
|    approx_kl            | 0.013491642 |
|    clip_fraction        | 0.201       |
|    clip_range           | 0.2         |
|    entropy_loss         | -4.22       |
|    explained_variance   | -0.0226     |
|    learning_rate        | 0.

In [1]:
import gymnasium as gym
from stable_baselines3 import PPO

# --- 1. Load the saved model ---
print("Loading model...")
model = PPO.load("ppo_hopper_gpu_ipynb")

# --- 2. Create a *single* environment with rendering ---
print("Creating environment to watch...")
env = gym.make("Hopper-v4", render_mode="human")

# --- 3. Run the agent ---
obs, _ = env.reset()
for _ in range(5000):
    # Get the agent's action
    action, _ = model.predict(obs, deterministic=True)

    # Perform the action in the environment
    obs, reward, terminated, truncated, info = env.step(action)

    # Reset if the episode ends
    if terminated or truncated:
        print("Episode finished. Resetting.")
        obs, _ = env.reset()

env.close()

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Loading model...


/home/jamesspectre007/anaconda3/envs/forcelearning/lib/python3.13/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
/home/jamesspectre007/anaconda3/envs/forcelearning/lib/python3.13/site-packages/gymnasium/envs/registration.py:512: DeprecationWarning: WARN: The environment Hopper-v4 is out of date. You should consider upgrading to version `v5`.
  logger.deprecation(


Creating environment to watch...
Episode finished. Resetting.
Episode finished. Resetting.
Episode finished. Resetting.
Episode finished. Resetting.
Episode finished. Resetting.
Episode finished. Resetting.
Episode finished. Resetting.
